In [ ]:
from pprint import pprint 

from desdeo.problem.testproblems.summer_cabin_electricity import generate_summer_cabin_electricity_data

import pandas as pd
import numpy as np


In [3]:
# Path to your uploaded file
file_path = "~/data/GUI_ENERGY_PRICES_202412312300-202512312300.csv"

# Load CSV
df = pd.read_csv(file_path)

# --- Try to detect column names ---
# Adjust these if needed after printing df.columns
print(df.columns)
print(df.head())
print(df.columns)

Index(['MTU (CET/CEST)', 'Area', 'Sequence', 'Day-ahead Price (EUR/MWh)',
       'Intraday Period (CET/CEST)', 'Intraday Price (EUR/MWh)'],
      dtype='object')
                              MTU (CET/CEST)    Area          Sequence  \
0  01/01/2025 00:00:00 - 01/01/2025 00:15:00  BZN|FI  Without Sequence   
1  01/01/2025 00:15:00 - 01/01/2025 00:30:00  BZN|FI  Without Sequence   
2  01/01/2025 00:30:00 - 01/01/2025 00:45:00  BZN|FI  Without Sequence   
3  01/01/2025 00:45:00 - 01/01/2025 01:00:00  BZN|FI  Without Sequence   
4  01/01/2025 01:00:00 - 01/01/2025 01:15:00  BZN|FI  Without Sequence   

   Day-ahead Price (EUR/MWh)  Intraday Period (CET/CEST)  \
0                       4.85                         NaN   
1                       4.85                         NaN   
2                       4.85                         NaN   
3                       4.85                         NaN   
4                       3.95                         NaN   

   Intraday Price (EUR/MWh)  
0 

In [4]:
# --- Extract start timestamp ---
raw_time = df["MTU (CET/CEST)"].str.split(" - ").str[0]

# --- Remove timezone labels like " (CET)" / " (CEST)" ---
clean_time = raw_time.str.replace(r" \(CET\)| \(CEST\)", "", regex=True)

# Parse datetime (day-first!)
df["timestamp"] = pd.to_datetime(clean_time, dayfirst=True)

df = df.set_index("timestamp")

# --- Keep only day-ahead price ---
df = df[["Day-ahead Price (EUR/MWh)"]].rename(columns={
    "Day-ahead Price (EUR/MWh)": "price"
})

# --- Filter June-August ---
df = df[(df.index.year == 2025) & (df.index.month >= 6) & (df.index.month <= 8)]

# --- Downsample to hourly (no averaging!) ---
df_hourly = df.resample("h").first()

prices = df_hourly["price"].values

print(df_hourly.head())
print(len(prices))

                     price
timestamp                 
2025-06-01 00:00:00  -0.10
2025-06-01 01:00:00  -0.11
2025-06-01 02:00:00  -0.20
2025-06-01 03:00:00  -0.21
2025-06-01 04:00:00  -0.10
2208


In [5]:
np.savez_compressed("prices_summer.npz", prices=prices)

In [8]:
price_data = np.load("datasets/prices_summer.npz")
print(price_data)

NpzFile 'datasets/prices_summer.npz' with keys: prices
